# JSON-LD vs TTL — same ontology, two syntaxes

Both formats encode the same RDF graph. seocho can read and write either, so you never have to pick a single representation forever — convert when you need to.

Short rule of thumb:

- **JSON-LD** — JSON with a few extra `@`-keys. Easy to hand-edit, easy to debug. The basic tutorials use this.
- **TTL (Turtle)** — compact triple syntax with prefix declarations. What FIBO, FOAF, Dublin Core, schema.org and other published vocabularies ship as.

This notebook builds one tiny ontology in code, writes it in both formats, prints them side by side, then proves the round-trip works.

## Setup

In [ ]:
import tempfile
from pathlib import Path

from seocho import Ontology, NodeDef, RelDef, P
from seocho.ontology import PropertyType

# Working directory for the two files we'll write/read.
tmp = Path(tempfile.mkdtemp(prefix="jsonld_vs_ttl_"))
print(f"Writing to: {tmp}")

## 1. Build a tiny ontology in Python

Two classes (`Company`, `Person`), one relationship (`EMPLOYS`). Same data we'll save to both formats below.

In [ ]:
onto = Ontology(
    name="company_demo",
    namespace="https://seocho.dev/demo/",
    description="A two-class ontology to illustrate JSON-LD vs TTL.",
    nodes={
        "Company": NodeDef(
            description="A registered business",
            aliases=["Firm", "Corporation"],
            properties={
                "name": P(type=PropertyType.STRING, unique=True),
                "sector": P(type=PropertyType.STRING),
                "headquarters": P(type=PropertyType.STRING),
            },
        ),
        "Person": NodeDef(
            description="An individual",
            properties={
                "name": P(type=PropertyType.STRING, unique=True),
                "title": P(type=PropertyType.STRING),
            },
        ),
    },
    relationships={
        "EMPLOYS": RelDef(
            source="Company",
            target="Person",
            description="Employment",
            cardinality="ONE_TO_MANY",
        ),
    },
)
print(f"Ontology: {onto.name}  |  {len(onto.nodes)} classes, {len(onto.relationships)} relationships")

## 2. Save as JSON-LD

JSON, with `@context` mapping prefixes to vocab URIs, `@id` for resource identity, `@type` for class membership. Looks familiar to any web developer.

In [ ]:
jsonld_path = tmp / "company_demo.jsonld"
onto.to_jsonld(jsonld_path)

print(jsonld_path.read_text())

## 3. Save as TTL (Turtle)

Each line is a single `subject predicate object .` triple. Prefix declarations at the top let you abbreviate URIs (`priv:Company` instead of the full one). The `;` chains predicates for the same subject.

In [ ]:
ttl_path = tmp / "company_demo.ttl"
onto.to_ttl(ttl_path)

print(ttl_path.read_text())

## 4. Verbosity side by side

In [ ]:
jsonld_lines = jsonld_path.read_text().splitlines()
ttl_lines    = ttl_path.read_text().splitlines()
print(f"JSON-LD : {len(jsonld_lines):>3} lines, {jsonld_path.stat().st_size:>4} bytes")
print(f"TTL     : {len(ttl_lines):>3} lines, {ttl_path.stat().st_size:>4} bytes")
print()
print("Both encode the same RDF graph. JSON-LD is more verbose because of")
print("JSON's brace/comma overhead; TTL is denser because of the prefix")
print("machinery and the ';' chaining.")

## 5. Round-trip check

Load each file back into an `Ontology` object and confirm we get the same shape — same classes, same relationships, same properties on `Company`.

In [ ]:
from_jsonld = Ontology.from_jsonld(jsonld_path)
from_ttl    = Ontology.from_ttl(ttl_path)

def shape(o):
    return {
        "classes": sorted(o.nodes.keys()),
        "rels": sorted(o.relationships.keys()),
        "company_props": sorted(o.nodes.get("Company", NodeDef()).properties.keys()),
    }

print(f"loaded from JSON-LD : {shape(from_jsonld)}")
print(f"loaded from TTL     : {shape(from_ttl)}")

## 6. Cross-format conversion is one method call

If you start with TTL (e.g. you downloaded a FIBO module) and want JSON-LD for a tool that prefers it, just call `to_jsonld()` on the loaded ontology. And vice versa.

In [ ]:
# Started as TTL? Save it as JSON-LD.
ttl_to_jsonld = tmp / "from_ttl.jsonld"
from_ttl.to_jsonld(ttl_to_jsonld)

# Started as JSON-LD? Save it as TTL.
jsonld_to_ttl = tmp / "from_jsonld.ttl"
from_jsonld.to_ttl(jsonld_to_ttl)

print(f"TTL -> JSON-LD: {ttl_to_jsonld.stat().st_size} bytes")
print(f"JSON-LD -> TTL: {jsonld_to_ttl.stat().st_size} bytes")
print()
print("Sizes match the originals (~58 lines / 1313 bytes for JSON-LD,")
print("~39 lines / 1077 bytes for TTL) because the data is preserved exactly.")

## What to take away

- **JSON-LD when you're authoring** — it's just JSON. You can hand-edit the file in any editor; syntax errors are obvious; every IDE highlights it.
- **TTL when you're consuming a published vocabulary** — FIBO ships as TTL, schema.org publishes TTL alongside JSON-LD, FOAF and Dublin Core are TTL-native. If you `wget` an ontology from a standards body, you'll get `.ttl`.
- **You don't have to commit forever** — `Ontology.from_jsonld()` / `Ontology.from_ttl()` for input, `Ontology.to_jsonld()` / `Ontology.to_ttl()` for output. Convert at the boundary.

When in doubt: start with JSON-LD. It has the fewer learning steps, and you can convert to TTL the moment you need to.